In [0]:
dbutils.widgets.text("catalog","fk")
catalog = dbutils.widgets.get("catalog")

#stores

In [0]:

from pyspark.sql import functions as F


stores = spark.table(f"{catalog}.bronze.stores")

silver_stores = (
    stores
    .select(
        F.upper(F.trim("store_id")).alias("store_id"),
        F.trim("store_name").alias("store_name"),
        F.initcap(F.trim("city")).alias("city"),
        F.trim("state").alias("state"),
        F.trim("region").alias("region"),
        F.to_date("opening_date", "yyyy-MM-dd")
         .alias("opening_date")
    )
)

(silver_stores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.stores"))

spark.sql(f"""
COMMENT ON TABLE {catalog}.silver.stores
IS 'Cleaned dark store master data'
""")


#customers

In [0]:
from pyspark.sql import functions as F

customers = spark.table(f"{catalog}.bronze.customers")

customers_clean = customers.select(
    F.upper(F.trim("customer_id")).alias("customer_id"),
    F.trim("full_name").alias("full_name"),
    F.lower(F.trim("email")).alias("email"),
    F.regexp_replace(
        F.trim("phone"), r"\D", ""
    ).alias("phone_digits"),
    F.initcap(F.trim("city")).alias("city"),
    F.upper(F.trim("tier")).alias("tier"),
    F.coalesce(
        F.to_date("signup_date", "yyyy-MM-dd"),
        F.to_date("signup_date", "dd/MM/yyyy")
    ).alias("signup_date"),
    F.to_date("change_effective_date").alias(
        "change_effective_date"
    )
)

customers_clean = customers_clean.withColumn(
    "phone",
    F.when(
        F.length("phone_digits") > 10,
        F.expr("right(phone_digits, 10)")
    ).otherwise(F.col("phone_digits"))
).drop("phone_digits")


(customers_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.silver.customers"))


#products

In [0]:

from pyspark.sql import functions as F

products = spark.table(f"{catalog}.bronze.products")

products_clean = products.select(
    F.upper(F.trim("product_id")).alias("product_id"),
    F.trim("product_name").alias("product_name"),
    F.trim("brand").alias("brand"),
    F.col("category.l1").alias("category_l1"),
    F.col("category.l2").alias("category_l2"),
    F.col("pack.size").cast("int").alias("pack_size"),
    F.col("pack.unit").alias("pack_unit"),
    F.col("mrp").cast("decimal(10,2)").alias("mrp"),
    F.col("selling_price").cast("decimal(10,2)")
     .alias("selling_price"),
    F.col("is_active").cast("boolean").alias("is_active"),
    F.col("tags").alias("tags"),
    F.to_date("snapshot_date").alias(
        "last_changed_snapshot_date"
    )
)

products_clean.createOrReplaceTempView("products_source")


In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS {catalog}.silver.products (
    product_id STRING,
    product_name STRING,
    brand STRING,
    category_l1 STRING,
    category_l2 STRING,
    pack_size INT,
    pack_unit STRING,
    mrp DECIMAL(10,2),
    selling_price DECIMAL(10,2),
    is_active BOOLEAN,
    tags ARRAY<STRING>,
    last_changed_snapshot_date DATE
)
USING DELTA;
        
          
""")

In [0]:
spark.sql(f"""
    ALTER TABLE {catalog}.silver.products
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
);      
""")


In [0]:
spark.sql(f"""
    
MERGE INTO {catalog}.silver.products AS t
USING products_source AS s
ON t.product_id = s.product_id

WHEN MATCHED AND (
       NOT (t.product_name <=> s.product_name)
    OR NOT (t.brand <=> s.brand)
    OR NOT (t.category_l1 <=> s.category_l1)
    OR NOT (t.category_l2 <=> s.category_l2)
    OR NOT (t.pack_size <=> s.pack_size)
    OR NOT (t.pack_unit <=> s.pack_unit)
    OR NOT (t.mrp <=> s.mrp)
    OR NOT (t.selling_price <=> s.selling_price)
    OR NOT (t.is_active <=> s.is_active)
    OR NOT (t.tags <=> s.tags)
)
THEN UPDATE SET
    t.product_name = s.product_name,
    t.brand = s.brand,
    t.category_l1 = s.category_l1,
    t.category_l2 = s.category_l2,
    t.pack_size = s.pack_size,
    t.pack_unit = s.pack_unit,
    t.mrp = s.mrp,
    t.selling_price = s.selling_price,
    t.is_active = s.is_active,
    t.tags = s.tags,
    t.last_changed_snapshot_date =
        s.last_changed_snapshot_date

WHEN NOT MATCHED THEN INSERT *;
      
          
          """)

#orders

In [0]:
# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {catalog}.silver.orders (
#     order_id STRING,
#     customer_id STRING,
#     store_id STRING,
#     order_ts TIMESTAMP,
#     order_date DATE,
#     order_status STRING,
#     payment_method STRING,
#     channel STRING,
#     promised_mins INT,
#     actual_mins INT,
#     delivery_fee DECIMAL(10,2),
#     updated_at TIMESTAMP
# )               
# """)

In [0]:
# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {catalog}.silver.order_items (
#     order_id INT,
#     line_number INT,
#     product_id STRING,
#     quantity INT,
#     unit_price DECIMAL(10,2),
#     discount_amount DECIMAL(10,2),
#     gross_amount DECIMAL(12,2),
#     net_amount DECIMAL(12,2)
# )    
# """)

In [0]:
# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {catalog}.silver.quarantine_orders (
#     order_id STRING,
#     reject_reason STRING,
#     updated_at TIMESTAMP,
#     raw_record STRING,
#     quarantined_at TIMESTAMP
# )     
          
# """)

In [0]:
# orders_df = spark.table(f"""{catalog}.bronze.orders""")

In [0]:
# from pyspark.sql import functions as F

# #standarization
# orders_df = orders_df.withColumn("order_id",F.initcap(F.trim("order_id"))) \
#     .withColumn("customer_id",F.initcap(F.trim("customer_id"))) \
#     .withColumn("store_id",F.initcap(F.trim("store_id"))) \
#     .withColumn("product_id",F.initcap(F.trim("product_id"))) \
#     .withColumn("order_status",F.upper(F.trim("order_status"))) \
#     .withColumn("payment_method",F.upper(F.trim("payment_method"))) \
#     .withColumn("channel",F.upper(F.trim(coalesce("channel","UNKNOWN")))) \
#     .withColumn("order_ts",F.to_timestamp(F.trim("order_ts"),"yyyy-MM-dd HH:mm:ss")) \
#     .withColumn("updated_at",F.to_timestamp())

# orders_df = orders_df.

# #deduplication
# w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc_nulls_last(),
#                                            F.col("_ingested_at").desc(), F.col("_source_file").desc())
# latest = (std.where("order_id IS NOT NULL").withColumn("rn", F.row_number().over(w))
#              .where("rn = 1").drop("rn"))

# #validation
# #quarantine

In [0]:
from pyspark.sql.functions import *

df = spark.read.table(f"{catalog}.bronze.orders")

df = (
    df.withColumn("order_id", col("order_id").cast("string"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("store_id", col("store_id").cast("string"))
    .withColumn("order_ts", try_to_timestamp(col("order_ts"), lit("yyyy-MM-dd HH:mm:ss")))
    .withColumn("order_date", to_date(col("order_ts")))
    .withColumn("order_status", trim(upper(col("order_status"))).cast("string"))
    .withColumn("payment_method", trim(upper(col("payment.method"))).cast("string"))
    .withColumn("channel", lit("Unknown"))
    .withColumn("promised_mins", col("delivery.promised_mins").cast("int"))
    .withColumn("actual_mins", col("delivery.actual_mins").cast("int"))
    .withColumn("delivery_fee", col("delivery.delivery_fee").cast("decimal(10,2)"))
    .withColumn("updated_at", try_to_timestamp(col("updated_at")))
)

df = df.drop(
    "ingested_at",
    "_source_file",
    "payment",
    "delivery",
    "items",
    "_rescued_data"
)

df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.silver.orders")

In [0]:
from pyspark.sql.functions import *

df = spark.read.table(f"{catalog}.bronze.orders")

df = df.withColumn("item", explode(col("items"))).drop("items")

df = (
    df.withColumn("order_id", col("order_id").cast("string"))
    .withColumn("product_id", col("item.product_id").cast("string"))
    .withColumn(
        "quantity",
        when(col("item.qty") > 0, col("item.qty")).otherwise(lit(None))
    )
    .withColumn(
        "unit_price",
        when(col("item.unit_price") > 0, col("item.unit_price")).otherwise(lit(None))
    )
    .withColumn(
        "discount_amount",
        when(col("item.discount") >= 0, col("item.discount"))
        .otherwise(lit(None))
    )
    .withColumn("gross_amount", col("quantity") * col("unit_price"))
    .withColumn(
        "net_amount",
        col("gross_amount") - coalesce(col("discount_amount"), lit(0))
    )
)

df = df.drop(
    "ingested_at",
    "_rescued_data",
    "_source_file",
    "payment",
    "delivery",
    "item",
    "updated_at",
    "order_ts",
    "order_status",
    "payment_method",
    "customer_id",
    "store_id"
)

df.write.option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.order_items")

In [0]:
from pyspark.sql import Window

def duplicate_order_ids():
    df = spark.read.table(f"{catalog}.bronze.orders")
    window = Window.partitionBy("order_id").orderBy(desc("updated_at"))
    df = df.withColumn("rn", row_number().over(window))
    df = df.filter(col("rn") == 1).drop("rn")
    return df

d = duplicate_order_ids()
d.count()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, broadcast

orders_df = duplicate_order_ids()

orders_df = (orders_df
    .withColumn("raw_record", F.to_json(F.struct(*orders_df.columns)))
    .withColumn("order_id", col("order_id").cast("string")))

valid_customers = (spark.read.table(f"{catalog}.silver.customers")
    .select(col("customer_id").cast("string").alias("valid_customer_id")).distinct())
valid_stores = (spark.read.table(f"{catalog}.silver.stores")
    .select(col("store_id").cast("string").alias("valid_store_id")).distinct())
valid_products = (spark.read.table(f"{catalog}.silver.products")      # active + inactive
    .select(col("product_id").cast("string").alias("valid_product_id")).distinct())

lines_df = (
    orders_df
    .filter(col("items").isNotNull() & (F.size("items") > 0))
    .select(col("order_id"), F.explode("items").alias("item"))
    .select(
        col("order_id"),
        col("item.product_id").cast("string").alias("line_product_id"),
        col("item.qty").alias("line_qty"),                 
        col("item.unit_price").alias("line_price"),
    )
    .join(broadcast(valid_products),
          col("line_product_id") == col("valid_product_id"), "left")
)

line_flags = (
    lines_df
    .withColumn("bad_product_i",
        (col("line_product_id").isNull() | col("valid_product_id").isNull()).cast("int"))
    # S7 (assumed): quantity must be a whole number > 0
    .withColumn("qty_int", F.expr("try_cast(line_qty as int)"))
    .withColumn("bad_qty_i",
        (col("qty_int").isNull() | (col("qty_int") <= 0)).cast("int"))
    # S8 (assumed): unit_price must be a number >= 0
    .withColumn("price_dec", F.expr("try_cast(line_price as decimal(18,2))"))
    .withColumn("bad_price_i",
        (col("price_dec").isNull() | (col("price_dec") < 0)).cast("int"))
    .groupBy("order_id")
    .agg(
        (F.max("bad_product_i") == 1).alias("bad_product"),
        (F.max("bad_qty_i") == 1).alias("bad_qty"),
        (F.max("bad_price_i") == 1).alias("bad_price"),
    )
)

ts_valid = F.coalesce(
    F.expr("try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss')"),
    F.expr("try_to_timestamp(order_ts, \"yyyy-MM-dd'T'HH:mm:ss\")")
).isNotNull()

flagged = (
    orders_df
    .join(broadcast(valid_customers),
          col("customer_id").cast("string") == col("valid_customer_id"), "left")
    .join(broadcast(valid_stores),
          col("store_id").cast("string") == col("valid_store_id"), "left")
    .join(line_flags, "order_id", "left")
)

reason = (
    F.when(col("customer_id").isNull(), "MISSING_CUSTOMER")                             
     .when(col("valid_customer_id").isNull(), "UNKNOWN_CUSTOMER")                      
     .when(col("store_id").isNull() | col("valid_store_id").isNull(), "UNKNOWN_STORE")   
     .when(col("order_ts").isNull() | ~ts_valid, "INVALID_TIMESTAMP")                    
     .when(col("items").isNull() | (F.size("items") == 0), "EMPTY_ORDER")                
     .when(col("bad_product"), "UNKNOWN_PRODUCT")                                       
     .when(col("bad_qty"), "INVALID_QUANTITY")                                         
     .when(col("bad_price"), "INVALID_PRICE")                                           
)

quarantine_orders_df = (
    flagged
    .withColumn("reject_reason", reason)
    .filter(col("reject_reason").isNotNull())
    .select(
        col("order_id").cast("string").alias("order_id"),
        col("reject_reason"),
        F.expr("try_to_timestamp(updated_at)").alias("updated_at"),
        col("raw_record"),
        F.current_timestamp().alias("quarantined_at"),
    )
)

(quarantine_orders_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.quarantine_orders"))

quarantine_orders_df.display()